# Exploratory statistics and aircraft profiles

Spearman analysis is descriptive. Freeze the date-selection rule before inspecting coefficients/p-values. Whole-period k-means is descriptive and is not fed into predictive models.

In [ ]:
from contextlib import closing
from pathlib import Path
import sqlite3
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = Path.cwd() if (Path.cwd() / "config.json").exists() else Path.cwd().parent
DB = ROOT / "data/processed/aircraft.sqlite"
if not DB.exists():
    raise RuntimeError("Collect real data and run src/prepare.py first.")
with closing(sqlite3.connect(DB)) as db:
    data = pd.read_sql_query("SELECT * FROM observations", db)
    airports = pd.read_sql_query("SELECT * FROM airports", db)
    audit = pd.read_sql_query("SELECT * FROM integration_audit", db)
data["observed_at_utc"] = pd.to_datetime(data.observed_at_utc, utc=True)
OUT = ROOT / "outputs"
(OUT / "figures").mkdir(parents=True, exist_ok=True)
(OUT / "tables").mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")


In [ ]:
from scipy.stats import spearmanr
# Provisional deterministic rule: second observed UTC date. Confirm/freeze after pilot.
# One earliest position per aircraft on that date, reducing repeated-aircraft dependence.
dates = sorted(data.date_utc.unique())
if len(dates)<2:
    raise RuntimeError("Collect at least two dates before this exploratory analysis.")
analysis_day = dates[1]
one = data[data.date_utc.eq(analysis_day)].sort_values("observed_at_utc").drop_duplicates("icao24")
rows=[]
for zone in ["near","far"]:
    subset = one[one.zone.eq(zone)]
    if len(subset)<20 or subset.geo_altitude.nunique()<2 or subset.velocity.nunique()<2:
        print(zone, "insufficient sample/variation; no test reported")
        continue
    result = spearmanr(subset.geo_altitude,subset.velocity)
    rows.append({"analysis_day_utc":analysis_day,"zone":zone,"n_aircraft":len(subset),
        "spearman_rho":result.statistic,"p_value":result.pvalue})
statistics = pd.DataFrame(rows)
display(statistics)
statistics.to_csv(OUT / "tables/exploratory_correlations.csv",index=False)
# P-values are not probabilities that a hypothesis is true, or that chance caused the data.
# Separate p-values do not test whether near/far relationships differ.
# Discuss remaining shared-weather/time dependence and aircraft mix.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
profile = data.groupby("icao24")[["geo_altitude","velocity"]].median().dropna()
if len(profile)<20:
    raise RuntimeError("Need at least 20 distinct aircraft profiles for clustering diagnostics.")
scaler = StandardScaler()
scaled = scaler.fit_transform(profile)
if np.unique(scaled,axis=0).shape[0]<6:
    raise RuntimeError("Too few distinct profiles to compare k=2..5.")
rows=[];fitted={}
for k in range(2,6):
    model = KMeans(n_clusters=k,n_init=10,random_state=42).fit(scaled)
    score = silhouette_score(scaled,model.labels_,sample_size=min(5000,len(scaled)),random_state=42)
    rows.append({"k":k,"inertia":model.inertia_,"silhouette":score})
    fitted[k]=model
scores=pd.DataFrame(rows)
display(scores)
k=int(scores.loc[scores.silhouette.idxmax(),"k"])
model=fitted[k]
profile["cluster"]=model.labels_
centroids = pd.DataFrame(scaler.inverse_transform(model.cluster_centers_),columns=["median_altitude_m","median_speed_m_s"])
display(centroids)
scores.to_csv(OUT / "tables/cluster_diagnostics.csv",index=False)
centroids.to_csv(OUT / "tables/cluster_centroids.csv",index_label="cluster")
sns.scatterplot(data=profile,x="geo_altitude",y="velocity",hue="cluster",palette="tab10")
plt.xlabel("Aircraft median geometric altitude (m)");plt.ylabel("Aircraft median ground speed (m/s)")
plt.tight_layout();plt.savefig(OUT / "figures/aircraft_profiles.png",dpi=160);plt.show()

## Group interpretation required

Interpret centroid values and sample sizes. Aircraft median profiles depend on where/when the aircraft was observed. They are not verified aircraft types. Compare silhouette and inertia rather than blindly choosing a pretty plot. Do not add a chi-squared test of speed bins versus these speed-derived clusters as independent evidence. The direct near/far adjusted relationship comparison requires additional common-support/interaction analysis.